<a href="https://colab.research.google.com/github/UniVR-DH/DKR-course/blob/main/L18-advanced/DuckPGQ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%pip install duckdb

In [3]:
# Install duckpgq libraries
import duckdb
conn = duckdb.connect(database=':memory:', read_only=False);
conn.install_extension("duckpgq", repository="community")
conn.load_extension("duckpgq")
print("DuckDB and DuckPGQ initialized successfully!")

In [4]:
conn.execute("CREATE TABLE Person AS SELECT * FROM 'https://gist.githubusercontent.com/Dtenwolde/2b02aebbed3c9638a06fda8ee0088a36/raw/8c4dc551f7344b12eaff2d1438c9da08649d00ec/person-sf0.003.csv';")
conn.execute("CREATE TABLE Person_knows_person AS SELECT * FROM 'https://gist.githubusercontent.com/Dtenwolde/81c32c9002d4059c2c3073dbca155275/raw/8b440e810a48dcaa08c07086e493ec0e2ec6b3cb/person_knows_person-sf0.003.csv';");

In [5]:
conn.execute("""
CREATE PROPERTY GRAPH snb
VERTEX TABLES (
    Person
  )
EDGE TABLES (
    Person_knows_person
        SOURCE KEY ( person1id ) REFERENCES Person ( id )
        DESTINATION KEY ( person2id ) REFERENCES Person ( id )
        LABEL Knows
  );
""")

In [11]:
firstName_param = 'Jan'
result = conn.execute(f"""
FROM GRAPH_TABLE(snb
    MATCH (a:Person WHERE a.firstName = '{firstName_param}')-[k:Knows]->(b:Person)
    COLUMNS (b.firstName)
);
""")

In [ ]:
FROM GRAPH_TABLE (snb
    MATCH p = ANY SHORTEST (a:Person WHERE a.firstName = 'Jan')-[k:knows]->+(b:Person)
    COLUMNS (path_length(p), b.firstName)
  )
ORDER BY firstName
LIMIT 5;

In [8]:
for row in result.fetchall():
    print(row)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

('Ali',)
('Otto',)
('Bryn',)
('Hans',)
